# 📘 Semaine 2 — Architecture hardware du STM32F103C6T6

**Cours :** Microcontrôleurs STM32F103C6T6  
**Durée :** 4h30 (1h30 cours + 1h30 atelier + 1h30 homework)  
**Enseignant :** ____________________  
**Étudiant :** ____________________  
**Date :** ____________________

---

## 🎯 Objectifs pédagogiques de la semaine

À la fin de cette semaine, l'étudiant sera capable de :

1. **Décrire** le bloc diagram interne du STM32F103C6T6.
2. **Identifier** les bus AHB, APB1 et APB2 et leurs périphériques.
3. **Lire** la mappe mémoire et localiser les zones Flash / SRAM / périphériques.
4. **Configurer** les horloges (HSI, HSE, PLL) pour atteindre 72 MHz.
5. **Analyser** `SystemClock_Config()` généré par CubeMX.

---

## 🧭 Guide de lecture du notebook

Avant de commencer, retenez 3 règles simples :

- **Adresse ≠ capacité** : une plage d’adressage peut être grande sans que le composant ait autant de mémoire.
- **Flash réelle ≠ alias de démarrage** : la vraie mémoire programme est en `0x0800_0000`, pas en `0x0000_0000`.
- **AHB/APB** : les bus ne sont pas “tous égaux” ; le cœur, la SRAM et les GPIO ne sont pas branchés au même niveau.

> 💡 Ce notebook est conçu pour aller du concept général vers le code concret. Si une notion semble abstraite, relisez la section mémoire et la section horloges avant de passer à la suite.

---

## 🗺️ Plan de la semaine

| Partie | Contenu | Durée |
|---|---|---|
| **A — Cours** | Activités 1 à 5 | 1h30 |
| **B — Atelier** | TP2 : Configuration horloge + GPIO | 1h30 |
| **C — Homework** | Exercices 1 à 3 | 1h30 |
| **D — Auto-évaluation** | Checklist finale | 5 min |

### ✅ À retenir dès le départ

- Le STM32F103C6T6 est un MCU Cortex-M3 cadencé à 72 MHz.
- La Flash programme réelle est de 32 Ko.
- La SRAM est de 10 Ko.
- Les registres périphériques sont mappés en mémoire dans la zone `0x4000_0000` et suivantes.
- L’arbre d’horloge est fondamental pour comprendre les timers, GPIO, ADC, USART et PLL.

---
# 🎓 PARTIE A — COURS INTÉGRÉ (1h30)

## 🔹 Activité 1 — Rappel & mise en contexte (10 min)

### 🔄 Rappel de la semaine 1
- Cœur **ARM Cortex-M3** @ 72 MHz
- Pipeline 3 étages (Fetch / Decode / Execute)
- **NVIC** : 16 niveaux de priorité
- Environnement : **STM32CubeIDE + ST-Link**

### ✍️ Questions flash (réponds en 2 min)
1. Combien d'étages a le pipeline Cortex-M3 ? → ...
2. Que signifie NVIC ? → ...
3. Quel registre contient le *Program Counter* ? → ...
4. Que fait `HAL_GPIO_TogglePin()` ? → ...

---

## 🔹 Activité 2 — Vue d'ensemble du STM32F103C6T6 (25 min)

### 📖 2.1 — Caractéristiques principales

| Paramètre | Valeur |
|---|---|
| **Cœur** | ARM Cortex-M3 @ 72 MHz |
| **Flash** | 32 Ko |
| **SRAM** | 10 Ko |
| **GPIO** | 37 (boîtier LQFP48) |
| **Timers** | 3 (TIM1, TIM2, TIM3) |
| **ADC** | 2 × 12 bits — 10 canaux |
| **USART** | 2 (USART1, USART2) |
| **SPI** | 2 (SPI1, SPI2) |
| **I2C** | 1 (I2C1) |
| **USB** | 1 × Full-Speed (12 Mbit/s) |
| **CAN** | 1 |
| **DMA** | 7 canaux (DMA1) |
| **Alimentation** | 2.0 V – 3.6 V |
| **Température** | −40 °C à +85 °C |
| **Boîtier** | LQFP48 (7 × 7 mm) |

### 📖 2.2 — Bloc diagram interne (simplifié)

```text
                     ┌─────────────────────┐
                     │   ARM Cortex-M3     │
                     │   72 MHz  +  NVIC   │
                     └──────────┬──────────┘
                                │
                     ┌──────────▼──────────┐
                     │     Bus Matrix      │
                     │      (AHB / APB)    │
                     └──┬─────┬─────┬──────┘
                        │     │     │
        ┌───────────────┘     │     └───────────────┐
        │                     │                     │
   ┌────▼─────┐         ┌─────▼─────┐         ┌─────▼─────┐
   │  Flash   │         │   SRAM    │         │   DMA1    │
   │  32 Ko   │         │   10 Ko   │         │  7 canaux │
   └──────────┘         └───────────┘         └─────┬─────┘
                                                    │
        ┌───────────────────────────────────────────┘
        │
   ┌────▼─────┐                    ┌──────────────┐
   │  APB2    │                    │    APB1      │
   │ 72 MHz   │                    │   36 MHz     │
   ├──────────┤                    ├──────────────┤
   │ TIM1     │                    │ TIM2, TIM3   │
   │ USART1   │                    │ USART2       │
   │ SPI1     │                    │ SPI2         │
   │ ADC1/2   │                    │ I2C1         │
   │ GPIO A–C │                    │ USB, CAN     │
   │ EXTI     │                    │ PWR, BKP     │
   └──────────┘                    └──────────────┘
```

### 🧠 À retenir visuellement

- **AHB** = bus principal rapide connecté au cœur et aux mémoires
- **APB2** = bus rapide pour les fonctions critiques (GPIO, ADC, USART1, TIM1)
- **APB1** = bus plus lent pour les périphériques plus standards (TIM2/3, I2C, USART2, SPI2, CAN)

> 💡 Les périphériques “critiques” sont souvent sur APB2 pour bénéficier d’une fréquence plus élevée.

### 📖 2.3 — Périphériques par bus

| Bus | Fréquence max | Périphériques |
|---|---|---|
| **AHB** | 72 MHz | Flash, SRAM, DMA, GPIO, RCC |
| **APB2** | 72 MHz | TIM1, USART1, SPI1, ADC1/2, EXTI, AFIO, GPIO |
| **APB1** | 36 MHz | TIM2, TIM3, USART2, SPI2, I2C1, USB, CAN, PWR |

> 💡 **Règle :** plus le bus est rapide, plus les périphériques critiques (timers avancés, ADC) y sont rattachés.

### 🧠 À retenir
- **AHB** : bus système rapide.
- **APB1** : bus lent (max 36 MHz).
- **APB2** : bus rapide (max 72 MHz).

---

## 🔹 Activité 3 — Mappe mémoire (25 min)

### 📖 3.1 — Organisation générale

Le STM32F103C6T6 utilise un espace d'adressage **32 bits** (4 Go), divisé en zones. Il est important de distinguer :

- la **zone d’adressage système** (espace mémoire vu par le microcontrôleur)
- la **capacité effective** de chaque mémoire (Flash, SRAM, périphériques)

| Adresse de début | Adresse de fin | Taille | Contenu |
|---|---|---|---|
| `0x0000_0000` | `0x1FFF_FFFF` | 512 Mo | **Alias de démarrage / système** (selon BOOT0/BOOT1) |
| `0x2000_0000` | `0x2000_27FF` | 10 Ko | **SRAM** |
| `0x4000_0000` | `0x4000_FFFF` | — | **Périphériques APB1** |
| `0x4001_0000` | `0x4001_FFFF` | — | **Périphériques APB2** |
| `0x4002_0000` | `0x4002_FFFF` | — | **Périphériques AHB** |
| `0x0800_0000` | `0x0800_7FFF` | 32 Ko | **Flash programme** |
| `0xE000_0000` | `0xE00F_FFFF` | — | **NVIC, SysTick, SCB** |

> ✅ Clarification importante : la plage `0x0000_0000` → `0x1FFF_FFFF` représente un espace d’adressage de 512 Mo, mais cette valeur ne correspond pas à la capacité réelle de la Flash du MCU. C’est une zone de démarrage / alias système. La mémoire programmée réelle du STM32F103C6T6 est la Flash située en `0x0800_0000` → `0x0800_7FFF`, soit 32 Ko.
>
> 📌 Règle mnémotechnique :
> - `0x0000_...` = démarrage / alias système
> - `0x0800_...` = Flash programme réelle
> - `0x2000_...` = SRAM
> - `0x4000_...` = périphériques
> - `0xE000_...` = NVIC / Cortex-M

### 🧠 Pourquoi cette distinction est importante ?

Un débutant peut penser que le MCU contient 512 Mo de mémoire Flash parce qu’il voit une plage de 512 Mo dans la mappe. Ce n’est pas le cas. La valeur 512 Mo décrit l’**espace d’adressage logique du système**, pas la capacité réelle de stockage du composant.

La capacité réelle est donnée par les spécifications du matériel :
- **Flash** : 32 Ko
- **SRAM** : 10 Ko

### 📖 3.2 — Adresses clés des registres

| Périphérique | Adresse de base |
|---|---|
| GPIOA | `0x4001_0800` |
| GPIOB | `0x4001_0C00` |
| GPIOC | `0x4001_1000` |
| RCC | `0x4002_1000` |
| FLASH (interface) | `0x4002_2000` |
| TIM1 | `0x4001_2C00` |
| TIM2 | `0x4000_0000` |
| TIM3 | `0x4000_0400` |
| ADC1 | `0x4001_2400` |
| USART1 | `0x4001_3800` |
| USART2 | `0x4000_4400` |
| I2C1 | `0x4000_5400` |
| SPI1 | `0x4001_3000` |
| SPI2 | `0x4000_3800` |
| EXTI | `0x4001_0400` |
| AFIO | `0x4001_0000` |

### 🐍 Simulation Python — Calcul d'adresses (10 min)

Le code suivant permet de vérifier la taille de la SRAM et de la Flash par calcul d'adresses.


### 📖 3.2 — Adresses clés des registres

| Périphérique | Adresse de base |
|---|---|
| GPIOA | `0x4001_0800` |
| GPIOB | `0x4001_0C00` |
| GPIOC | `0x4001_1000` |
| RCC | `0x4002_1000` |
| FLASH (interface) | `0x4002_2000` |
| TIM1 | `0x4001_2C00` |
| TIM2 | `0x4000_0000` |
| TIM3 | `0x4000_0400` |
| ADC1 | `0x4001_2400` |
| USART1 | `0x4001_3800` |
| USART2 | `0x4000_4400` |
| I2C1 | `0x4000_5400` |
| SPI1 | `0x4001_3000` |
| SPI2 | `0x4000_3800` |
| EXTI | `0x4001_0400` |
| AFIO | `0x4001_0000` |

### 🐍 Simulation Python — Calcul d'adresses (10 min)

Le code suivant permet de vérifier la taille de la SRAM et de la Flash par calcul d'adresses.

In [1]:
# ============================================================
# Calcul des tailles mémoire du STM32F103C6T6
# ============================================================

def taille_ko(debut, fin):
    """Retourne la taille en Ko entre deux adresses incluses."""
    return (fin - debut + 1) / 1024

# Flash : 0x0800_0000 à 0x0800_7FFF (32 Ko)
flash_debut = 0x08000000
flash_fin   = 0x08007FFF

# SRAM : 0x2000_0000 à 0x2000_27FF (10 Ko)
sram_debut  = 0x20000000
sram_fin    = 0x200027FF

print("📦 MÉMOIRES DU STM32F103C6T6")
print("-" * 45)
print(f"Flash : 0x{flash_debut:08X} → 0x{flash_fin:08X}  = {taille_ko(flash_debut, flash_fin):.0f} Ko")
print(f"SRAM  : 0x{sram_debut:08X} → 0x{sram_fin:08X}  = {taille_ko(sram_debut, sram_fin):.0f} Ko")

# Adresses de base de quelques périphériques
periph = {
    "GPIOA":  0x40010800,
    "GPIOC":  0x40011000,
    "RCC":    0x40021000,
    "TIM2":   0x40000000,
    "USART1": 0x40013800,
    "ADC1":   0x40012400,
}

print("\n🔧 ADRESSES DE BASE DES PÉRIPHÉRIQUES")
print("-" * 45)
for nom, adr in periph.items():
    bus = "APB1" if adr < 0x40010000 else ("APB2" if adr < 0x40020000 else "AHB")
    print(f"{nom:<8} : 0x{adr:08X}  ({bus})")

📦 MÉMOIRES DU STM32F103C6T6
---------------------------------------------
Flash : 0x08000000 → 0x08007FFF  = 32 Ko
SRAM  : 0x20000000 → 0x200027FF  = 10 Ko

🔧 ADRESSES DE BASE DES PÉRIPHÉRIQUES
---------------------------------------------
GPIOA    : 0x40010800  (APB2)
GPIOC    : 0x40011000  (APB2)
RCC      : 0x40021000  (AHB)
TIM2     : 0x40000000  (APB1)
USART1   : 0x40013800  (APB2)
ADC1     : 0x40012400  (APB2)


---

## 🔹 Activité 4 — Horloges (25 min)

### 📖 4.1 — Sources d'horloge

| Source | Type | Fréquence | Précision | Usage |
|---|---|---|---|---|
| **HSI** | Interne RC | 8 MHz | ±1 % | Démarrage rapide |
| **HSE** | Externe quartz | 4–16 MHz (souvent 8 MHz) | ±20 ppm | Horloge précise |
| **PLL** | Multiplicateur | ×2 à ×16 | — | Atteindre 72 MHz |
| **LSI** | Interne RC | ~40 kHz | Faible | IWDG, RTC |
| **LSE** | Externe quartz | 32.768 kHz | Très haute | RTC |

### 📖 4.2 — Arbre d'horloge (atteindre 72 MHz avec HSE 8 MHz)

```
HSE (8 MHz) ──▶ PREDIV (/1) ──▶ PLL ×9 ──▶ SYSCLK (72 MHz)
                                              │
              ┌───────────────────────────────┼─────────────────────────────┐
              │                               │                             │
              ▼                               ▼                             ▼
           AHB /1                        APB1 /2                       APB2 /1
           HCLK = 72 MHz                 PCLK1 = 36 MHz                PCLK2 = 72 MHz
              │                               │                             │
              ▼                               ▼                             ▼
           Cortex-M3                       TIM2,3 ×2                     TIM1 ×1
           Flash, SRAM                     USART2, I2C1                  USART1, SPI1
           DMA, GPIO                       SPI2, USB, CAN                ADC, EXTI
```

> ⚠️ **Attention** : le timer reçoit l'horloge ×2 si le prescaler APB ≠ 1.  
> Exemple : TIM2 sur APB1 → horloge réelle = 72 MHz (36 × 2), pas 36 MHz.

### 🐍 Simulation Python — Calculs d'horloge (10 min)

Comprendre l'impact des prescalers sur les fréquences finales.

In [ ]:
# ============================================================
# Simulateur d'arbre d'horloge STM32F103
# ============================================================

def clock_tree(source, source_mhz, prediv=1, pll_mul=9,
               ahb_prescaler=1, apb1_prescaler=2, apb2_prescaler=1):
    """Calcule les fréquences du STM32F103."""
    pll_in   = source_mhz / prediv
    sysclk   = pll_in * pll_mul
    hclk     = sysclk / ahb_prescaler
    pclk1    = hclk / apb1_prescaler
    pclk2    = hclk / apb2_prescaler
    # Horloge timer : ×2 si prescaler APB ≠ 1
    tim_apb1 = pclk1 if apb1_prescaler == 1 else pclk1 * 2
    tim_apb2 = pclk2 if apb2_prescaler == 1 else pclk2 * 2
    return {
        "Source":      f"{source} {source_mhz} MHz",
        "PLL input":   f"{pll_in:.3f} MHz",
        "SYSCLK":      f"{sysclk:.1f} MHz",
        "HCLK (AHB)":  f"{hclk:.1f} MHz",
        "PCLK1 (APB1)":f"{pclk1:.1f} MHz",
        "PCLK2 (APB2)":f"{pclk2:.1f} MHz",
        "TIM APB1":    f"{tim_apb1:.1f} MHz",
        "TIM APB2":    f"{tim_apb2:.1f} MHz",
    }

def afficher(config):
    for k, v in config.items():
        print(f"  {k:<14} : {v}")

print("═" * 50)
print("CONFIGURATION 1 — HSE 8 MHz, PLL ×9  → 72 MHz")
print("═" * 50)
afficher(clock_tree("HSE", 8, prediv=1, pll_mul=9))

print()
print("═" * 50)
print("CONFIGURATION 2 — HSI 8 MHz, PLL ×9  → 72 MHz")
print("═" * 50)
afficher(clock_tree("HSI", 8, prediv=1, pll_mul=9))

print()
print("═" * 50)
print("CONFIGURATION 3 — HSE 4 MHz, PLL ×16 → 64 MHz")
print("═" * 50)
afficher(clock_tree("HSE", 4, prediv=1, pll_mul=16))

### 📖 4.3 — Registres RCC essentiels

| Registre | Rôle |
|---|---|
| **RCC_CR** | Contrôle des oscillateurs (HSION, HSEON, PLLON, READY) |
| **RCC_CFGR** | Configuration PLL, prescalers, source SYSCLK |
| **RCC_CIR** | Interruptions horloge |
| **RCC_APB2ENR** | Activation horloges APB2 (GPIOA, USART1, ADC1…) |
| **RCC_APB1ENR** | Activation horloges APB1 (TIM2, USART2, I2C1…) |
| **RCC_AHBENR** | Activation horloges AHB (DMA, SRAM, FLITF) |

> 🔑 **Important :** avant d'utiliser un périphérique, il faut **activer son horloge** via `__HAL_RCC_xxx_CLK_ENABLE()`.
>
> 🧠 **Mnémonique :**
> - **RCC_APB2ENR** = périphériques rapides / critiques
> - **RCC_APB1ENR** = périphériques plus lents / standards
> - **RCC_AHBENR** = bus système et mémoires

### 🧩 Question de synthèse

Pourquoi faut-il activer l’horloge d’un GPIO avant de lui écrire des valeurs ?

Réponse : parce que sans horloge, le périphérique n’est pas alimenté logique par le bus, et ses registres restent inactifs ou non accessibles.

---

## 🔹 Activité 5 — QCM formatif (10 min)

Réponds sans regarder tes notes.

**1. La fréquence maximale du bus APB1 est :**  
A. 18 MHz  
B. 36 MHz  
C. 72 MHz  
D. 144 MHz

**2. Le bus qui relie le cœur au Flash et à la SRAM est :**  
A. APB1  
B. APB2  
C. AHB  
D. USB

**3. Pour obtenir 72 MHz à partir de HSE 8 MHz, on utilise :**  
A. PLL ×4  
B. PLL ×6  
C. PLL ×9  
D. PLL ×16

**4. La SRAM du STM32F103C6T6 fait :**  
A. 4 Ko  
B. 10 Ko  
C. 20 Ko  
D. 32 Ko

**5. L'adresse de base de GPIOC est :**  
A. `0x4001_0800`  
B. `0x4001_0C00`  
C. `0x4001_1000`  
D. `0x4002_1000`

**6. Lequel de ces périphériques est sur APB2 ?**  
A. TIM2  
B. USART2  
C. I2C1  
D. USART1

### ✅ Corrigé du QCM formatif

| Q | Réponse | Explication |
|---|---|---|
| 1 | **B — 36 MHz** | APB1 = HCLK / 2 max |
| 2 | **C — AHB** | Bus système haute performance |
| 3 | **C — PLL ×9** | 8 × 9 = 72 MHz |
| 4 | **B — 10 Ko** | 0x2000_0000 → 0x2000_27FF |
| 5 | **C — 0x4001_1000** | GPIOC sur APB2 |
| 6 | **D — USART1** | USART1 est sur APB2 |

**Mon score : ___ / 6**

---

# 🛠️ PARTIE B — ATELIER / TP (1h30)

## 🧪 TP2 — Configuration horloge + GPIO via CubeMX

### 🎯 Objectif
Configurer les horloges du STM32F103C6T6 à **72 MHz** via HSE + PLL, activer les GPIO, et vérifier la configuration à l'oscilloscope.

### 📋 Tâches à réaliser (par binôme)

| # | Tâche | Durée | Livrable |
|---|---|---|---|
| 1 | Ouvrir CubeMX, sélectionner `STM32F103C6Tx` | 10 min | Capture |
| 2 | Activer HSE (Crystal/Ceramic Resonator) | 10 min | Capture |
| 3 | Configurer PLL ×9 → SYSCLK = 72 MHz | 15 min | Capture Clock Tree |
| 4 | Configurer PC13 (LED) et PA0 (bouton) | 10 min | Capture Pinout |
| 5 | Générer le code et inspecter `SystemClock_Config()` | 20 min | Code |
| 6 | Flasher et vérifier à l'oscilloscope sur MCO (PA8) | 15 min | Mesure |
| 7 | Rédiger le compte-rendu | 10 min | CR |

### ⚙️ Code généré par CubeMX — `SystemClock_Config()`

In [ ]:
/* ============================================================
   TP2 - SystemClock_Config() avec HSE 8 MHz + PLL ×9 = 72 MHz
   Généré par STM32CubeMX - fichier Core/Src/main.c
   ============================================================ */

void SystemClock_Config(void)
{
    RCC_OscInitTypeDef RCC_OscInitStruct = {0};
    RCC_ClkInitTypeDef RCC_ClkInitStruct = {0};

    /* --- Configuration des oscillateurs --- */
    RCC_OscInitStruct.OscillatorType       = RCC_OSCILLATORTYPE_HSE;
    RCC_OscInitStruct.HSEState             = RCC_HSE_ON;
    RCC_OscInitStruct.HSEPredivValue       = RCC_HSE_PREDIV_DIV1;
    RCC_OscInitStruct.HSIState             = RCC_HSI_ON;
    RCC_OscInitStruct.PLL.PLLState         = RCC_PLL_ON;
    RCC_OscInitStruct.PLL.PLLSource        = RCC_PLLSOURCE_HSE;
    RCC_OscInitStruct.PLL.PLLMUL           = RCC_PLL_MUL9;  // 8 MHz × 9 = 72 MHz
    if (HAL_RCC_OscConfig(&RCC_OscInitStruct) != HAL_OK)
    {
        Error_Handler();
    }

    /* --- Configuration des bus et prescalers --- */
    RCC_ClkInitStruct.ClockType      = RCC_CLOCKTYPE_HCLK   | RCC_CLOCKTYPE_SYSCLK
                                     | RCC_CLOCKTYPE_PCLK1  | RCC_CLOCKTYPE_PCLK2;
    RCC_ClkInitStruct.SYSCLKSource   = RCC_SYSCLKSOURCE_PLLCLK;
    RCC_ClkInitStruct.AHBCLKDivider  = RCC_SYSCLK_DIV1;   // HCLK  = 72 MHz
    RCC_ClkInitStruct.APB1CLKDivider = RCC_HCLK_DIV2;     // PCLK1 = 36 MHz
    RCC_ClkInitStruct.APB2CLKDivider = RCC_HCLK_DIV1;     // PCLK2 = 72 MHz
    if (HAL_RCC_ClockConfig(&RCC_ClkInitStruct, FLASH_LATENCY_2) != HAL_OK)
    {
        Error_Handler();
    }
}

### 🔍 Analyse du code

| Ligne clé | Signification |
|---|---|
| `RCC_OscInitStruct.HSEState = RCC_HSE_ON` | Active l'oscillateur externe HSE |
| `PLL.PLLSource = RCC_PLLSOURCE_HSE` | La PLL prend HSE en entrée |
| `PLL.PLLMUL = RCC_PLL_MUL9` | Multiplicateur ×9 → 72 MHz |
| `SYSCLKSource = RCC_SYSCLKSOURCE_PLLCLK` | SYSCLK provient de la PLL |
| `APB1CLKDivider = RCC_HCLK_DIV2` | APB1 = 72 / 2 = 36 MHz |
| `FLASH_LATENCY_2` | 2 cycles d'attente Flash pour 72 MHz |

> ⚠️ **FLASH_LATENCY** : à 72 MHz, la Flash ne peut pas suivre le CPU sans états d'attente.

### ⚙️ Activation des horloges GPIO

Avant d'utiliser un GPIO, il faut activer son horloge.

In [ ]:
/* --- Activation des horloges GPIO (à placer avant MX_GPIO_Init) --- */

__HAL_RCC_GPIOA_CLK_ENABLE();
__HAL_RCC_GPIOB_CLK_ENABLE();
__HAL_RCC_GPIOC_CLK_ENABLE();
__HAL_RCC_AFIO_CLK_ENABLE();     // Nécessaire pour EXTI / remap

/* --- Configuration de la sortie MCO (PA8) pour mesurer SYSCLK --- */
/* MCO = Microcontroller Clock Output : permet de sortir SYSCLK/HSE/HSI sur une pin */
HAL_RCC_MCOConfig(RCC_MCO1, RCC_MCO1SOURCE_SYSCLK, RCC_MCODIV_1);

### 📝 Compte-rendu de TP2

**Nom :** __________________  **Prénom :** __________________  **Binôme :** __________________

**1. Configuration CubeMX**
- Source d'horloge : ☐ HSI  ☐ HSE
- Fréquence HSE : ... MHz
- Multiplicateur PLL : ×...
- SYSCLK : ... MHz
- HCLK : ... MHz
- PCLK1 : ... MHz
- PCLK2 : ... MHz

**2. Capture d'écran du Clock Tree**
> *(à joindre)*

**3. Mesure à l'oscilloscope sur MCO (PA8)**
- Fréquence mesurée : ... MHz
- Écart avec la valeur attendue : ... %

**4. Flash Latency observée :**
- Nombre de wait states : ...
- Justification : ...

**5. Problèmes rencontrés**
- ...

**6. Solutions apportées**
- ...

### 🧪 Exercice bonus

Modifier la configuration pour obtenir **48 MHz** au lieu de 72 MHz, en utilisant HSE 8 MHz et un PLL ×6.

**Questions :**
1. Quelles sont les nouvelles fréquences HCLK / PCLK1 / PCLK2 ?
2. Quel prescaler AHB faut-il pour respecter les limites ?
3. Peut-on laisser APB1 à /2 ? Pourquoi ?

In [ ]:
# Corrigé bonus — Calculs pour 48 MHz
config_48 = clock_tree("HSE", 8, prediv=1, pll_mul=6,
                       ahb_prescaler=1, apb1_prescaler=2, apb2_prescaler=1)
print("═" * 50)
print("CONFIGURATION BONUS — HSE 8 MHz, PLL ×6  → 48 MHz")
print("═" * 50)
afficher(config_48)

---

# 🏠 PARTIE C — HOMEWORK (1h30)

## 📚 Exercices à rendre

### 🧩 Exercice 1 — Mappe mémoire (30 min)

Dessiner (à la main ou avec un outil comme **draw.io**) la **mappe mémoire complète** du STM32F103C6T6, en indiquant :
- Les zones Code, SRAM, Périphériques
- Les adresses de début et de fin
- Les tailles en Ko
- Les adresses de base de **5 périphériques** au choix

👉 Insérer le schéma ci-dessous (image ou ASCII).

### ✍️ Mappe mémoire personnalisée

```
(à compléter)
```

**Tableau récapitulatif :**

| Zone | Adresse début | Adresse fin | Taille |
|---|---|---|---|
| Flash | ... | ... | ... |
| SRAM | ... | ... | ... |
| APB1 | ... | ... | ... |
| APB2 | ... | ... | ... |
| AHB | ... | ... | ... |

### 🧩 Exercice 2 — Calculs PLL (30 min)

Trouver **3 configurations** permettant d'obtenir **SYSCLK = 72 MHz**, en variant la source d'entrée (HSI ou HSE) et le multiplicateur.

**Contraintes :**
- La PLL doit avoir une entrée entre **4 et 16 MHz**.
- Le multiplicateur doit être entre **×2 et ×16**.
- La sortie PLL ne doit pas dépasser **72 MHz**.

In [ ]:
# ============================================================
# Exercice 2 — Recherche de configurations PLL valides
# ============================================================

def chercher_configs(cible=72, pll_min=4, pll_max=16,
                     mul_min=2, mul_max=16):
    """Cherche toutes les combinaisons (source, prediv, mul) donnant SYSCLK = cible."""
    resultats = []
    sources = [("HSI", 8), ("HSE", 8), ("HSE", 4), ("HSE", 12), ("HSE", 16)]
    predivs = [1, 2]
    for nom, f in sources:
        for prediv in predivs:
            pll_in = f / prediv
            if not (pll_min <= pll_in <= pll_max):
                continue
            for mul in range(mul_min, mul_max + 1):
                sysclk = pll_in * mul
                if abs(sysclk - cible) < 0.01:
                    resultats.append((nom, f, prediv, pll_in, mul, sysclk))
    return resultats

print("🔎 Configurations valides pour SYSCLK = 72 MHz")
print("-" * 70)
print(f"{'Source':<8}{'F (MHz)':<10}{'PREDIV':<8}{'PLL in':<10}{'×':<4}{'SYSCLK'}")
print("-" * 70)
for nom, f, prediv, pll_in, mul, sysclk in chercher_configs():
    print(f"{nom:<8}{f:<10}{prediv:<8}{pll_in:<10}{mul:<4}{sysclk}")

**✍️ Mes 3 configurations choisies :**

| # | Source | PREDIV | PLL in | Multiplicateur | SYSCLK |
|---|---|---|---|---|---|
| 1 | ... | ... | ... | ×... | 72 MHz |
| 2 | ... | ... | ... | ×... | 72 MHz |
| 3 | ... | ... | ... | ×... | 72 MHz |

### 🧩 Exercice 3 — Lecture du RM0008 (30 min)

Lire les chapitres suivants du **Reference Manual RM0008** :
- **Chapitre 7** : Reset and clock control (RCC)
- **Chapitre 9** : General-purpose and alternate-function I/Os (GPIO)

Répondre aux questions suivantes :

1. Que se passe-t-il si on active un périphérique sans activer son horloge ?
2. Que signifie le bit **HSERDY** dans `RCC_CR` ?
3. Quelle est la différence entre **`RCC_APB2ENR`** et **`RCC_APB1ENR`** ?
4. Combien de registres de configuration GPIO y a-t-il par port ? Les nommer.
5. Que fait le registre **`BSRR`** et pourquoi est-il préféré à `ODR` ?

### ✍️ Réponses — Exercice 3

1. ...
2. ...
3. ...
4. ...
5. ...

---

## 🧮 Exercice supplémentaire — Simulateur de prescalers (optionnel)

Complète la cellule suivante pour calculer les fréquences finales en fonction de paramètres donnés.

In [ ]:
# Complète la fonction ci-dessous

def calcul_horloges(hse_mhz, pll_mul, ahb_div, apb1_div, apb2_div):
    """
    Retourne un dict avec SYSCLK, HCLK, PCLK1, PCLK2 en MHz.
    - hse_mhz  : fréquence HSE en MHz
    - pll_mul  : multiplicateur PLL
    - ahb_div  : diviseur AHB (1, 2, 4, 8, 16, 64, 128, 256, 512)
    - apb1_div : diviseur APB1 (1, 2, 4, 8, 16)
    - apb2_div : diviseur APB2 (1, 2, 4, 8, 16)
    """
    # TODO
    pass

# Tests
# print(calcul_horloges(8, 9, 1, 2, 1))
# print(calcul_horloges(8, 6, 1, 2, 1))
# print(calcul_horloges(8, 9, 1, 4, 2))

In [ ]:
# ✅ Corrigé
def calcul_horloges(hse_mhz, pll_mul, ahb_div, apb1_div, apb2_div):
    sysclk = hse_mhz * pll_mul
    hclk   = sysclk / ahb_div
    pclk1  = hclk / apb1_div
    pclk2  = hclk / apb2_div
    return {
        "SYSCLK": sysclk,
        "HCLK":   hclk,
        "PCLK1":  pclk1,
        "PCLK2":  pclk2,
    }

for params in [(8, 9, 1, 2, 1), (8, 6, 1, 2, 1), (8, 9, 1, 4, 2)]:
    hse, mul, ahb, apb1, apb2 = params
    print(f"HSE={hse} MHz, PLL ×{mul}, AHB /{ahb}, APB1 /{apb1}, APB2 /{apb2}")
    for k, v in calcul_horloges(*params).items():
        print(f"  {k:<8} = {v} MHz")
    print()

---
# ✅ PARTIE D — AUTO-ÉVALUATION Semaine 2

Coche ce que tu maîtrises.

- [ ] Je connais les caractéristiques du STM32F103C6T6 (Flash, SRAM, GPIO, ADC…).
- [ ] Je sais distinguer AHB, APB1, APB2.
- [ ] Je connais les adresses de base de GPIOA/B/C et RCC.
- [ ] Je sais calculer la taille de la Flash et de la SRAM.
- [ ] Je connais les sources d'horloge (HSI, HSE, PLL, LSI, LSE).
- [ ] Je sais configurer PLL ×9 pour 72 MHz.
- [ ] Je comprends le rôle des prescalers AHB/APB1/APB2.
- [ ] Je sais lire `SystemClock_Config()`.
- [ ] J'ai activé les horloges GPIO avec `__HAL_RCC_xxx_CLK_ENABLE()`.
- [ ] J'ai mesuré SYSCLK sur MCO (PA8).
- [ ] J'ai rédigé mon compte-rendu de TP2.
- [ ] J'ai complété la mappe mémoire.
- [ ] J'ai trouvé 3 configurations PLL pour 72 MHz.
- [ ] J'ai lu les chapitres 7 et 9 du RM0008.

### 📊 Mon score : ___ / 14

| Score | Interprétation |
|---|---|
| 12–14 | ✅ Prêt pour la S3 (GPIO) |
| 8–11 | ⚠️ Revoir les points manquants |
| < 8 | 🔁 Reprendre les activités 2 à 5 |

---
# ✅ PARTIE D — AUTO-ÉVALUATION Semaine 2

Coche ce que tu maîtrises.

- [ ] Je connais les caractéristiques du STM32F103C6T6 (Flash, SRAM, GPIO, ADC…).
- [ ] Je sais distinguer AHB, APB1, APB2.
- [ ] Je connais les adresses de base de GPIOA/B/C et RCC.
- [ ] Je sais calculer la taille de la Flash et de la SRAM.
- [ ] Je connais les sources d'horloge (HSI, HSE, PLL, LSI, LSE).
- [ ] Je sais configurer PLL ×9 pour 72 MHz.
- [ ] Je comprends le rôle des prescalers AHB/APB1/APB2.
- [ ] Je sais lire `SystemClock_Config()`.
- [ ] J'ai activé les horloges GPIO avec `__HAL_RCC_xxx_CLK_ENABLE()`.
- [ ] J'ai mesuré SYSCLK sur MCO (PA8).
- [ ] J'ai rédigé mon compte-rendu de TP2.
- [ ] J'ai complété la mappe mémoire.
- [ ] J'ai trouvé 3 configurations PLL pour 72 MHz.
- [ ] J'ai lu les chapitres 7 et 9 du RM0008.

### 📊 Mon score : ___ / 14

| Score | Interprétation |
|---|---|
| 12–14 | ✅ Prêt pour la S3 (GPIO) |
| 8–11 | ⚠️ Revoir les points manquants |
| < 8 | 🔁 Reprendre les activités 2 à 5 |

### 🧠 Réflexion finale

Avant de passer à la semaine 3, demande-toi :
- Quelle est la vraie mémoire Flash du MCU ?
- Quel bus relie le cœur à la SRAM ?
- Quelle est la configuration exacte pour obtenir 72 MHz ?
- Comment activer l’horloge d’un périphérique avant de l’utiliser ?

Si tu sais répondre à ces 4 questions, tu as bien compris la base de la semaine 2.